In [36]:
import torch
import torch.nn as nn

In [37]:
class LSTMModel(nn.Module):
    
    def __init__(self, input_size, hidden_size, output_size):
        
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden
    
    def init_hidden(self, batch_size):
        return (torch.zeros(1, batch_size, self.hidden_size), 
                torch.zeros(1, batch_size, self.hidden_size)
                )

In [38]:
input_size = 7
hidden_size = 10
output_size = 7
model = LSTMModel(input_size, hidden_size, output_size)

In [39]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [40]:
sequence = "decimal"
chars = sorted(list(set(sequence)))
char_to_idx = {ch:idx for idx, ch in enumerate(chars)}
idx_to_char = {idx:ch for ch, idx in char_to_idx.items()}

In [41]:
input_seq = [char_to_idx[ch] for ch in sequence[:-1]]
target_seq = [char_to_idx[ch] for ch in sequence[1:]]
input_tensor = torch.eye(len(chars))[input_seq].unsqueeze(0)
target_tensor = torch.tensor(target_seq).unsqueeze(0)

In [42]:
num_epochs = 100

for epoch in range(num_epochs):
    
    hidden = model.init_hidden(batch_size=1)
    output, hidden = model(input_tensor, hidden)

    loss = criterion(output.view(-1, output_size), target_tensor.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

test_seq = "ma"
test_input = torch.eye(len(chars))[[char_to_idx[ch] for ch in test_seq]].unsqueeze(0)
hidden = model.init_hidden(batch_size=1)
output, hidden = model(test_input, hidden)

predicted_idx = torch.argmax(output, dim=2).squeeze().tolist()
predicted_chars = ''.join(idx_to_char[idx] for idx in predicted_idx)
print(f"Predicted Characters: {predicted_chars}")

Epoch [10/100], Loss: 1.8201
Epoch [20/100], Loss: 1.5258
Epoch [30/100], Loss: 1.0975
Epoch [40/100], Loss: 0.6586
Epoch [50/100], Loss: 0.3735
Epoch [60/100], Loss: 0.2023
Epoch [70/100], Loss: 0.1074
Epoch [80/100], Loss: 0.0636
Epoch [90/100], Loss: 0.0428
Epoch [100/100], Loss: 0.0315
Predicted Characters: el
